# Thinking in DAX, building in Snowflake

Companion notebook for a walkthrough of the DAX context-manipulation scenarios in Mateusz Mossakowski's article
[Thinking in DAX, building in Databricks: Can metric views measures handle context manipulation?](https://www.linkedin.com/pulse/thinking-dax-building-databricks-can-metric-views-mateusz-mossakowski-3gu6f/),
rebuilt on a Snowflake semantic view.

Same model as the article: a sales fact (`sales_qty`) with Product, Store (Store ID, Customer, Country), and Calendar
dimensions. Extras: a 4-4-5 fiscal calendar and a `net_profit` measure that goes negative.

**Run the cells in order.** The setup cells create about 2.4M rows of deterministic mock data and the semantic view
in a schema you choose. Every number matches the post.

**Feature status:** `time_shift` is in Private Preview. LOD metrics (`FIXED` / `EXCLUDE`) and many-to-many
relationships are in Public Preview. Everything else uses generally available semantic view features.

## Setup

### 1. Choose where to build

By default this uses your current database and creates a schema named `DAX_CONTEXT_DEMO`. Change either value
if you like. Every later cell refers to objects through `FQ` (`<database>.<schema>`) and `SV` (the semantic view).

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = session.get_current_database()   # or set explicitly, e.g. "MY_DB"
SCHEMA = "DAX_CONTEXT_DEMO"

FQ = f"{DATABASE}.{SCHEMA}"
SV = f"{FQ}.DAX_CONTEXT_SV"

session.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}").collect()
print(f"Building in {FQ}")

### 2. Create the mock data

A star schema shaped like the article's: daily sales for 2020–2022, 55 stores in 3 countries (11 customers, which
are store chains within a country), and 40 products in 4 categories. The calendar includes 4-4-5 fiscal periods and
runs one year past the data so that "last year" rows have somewhere to land. Four small tables
(`DIM_COUNTRY`, `DIM_BRAND`, `DIM_CATEGORY`, `DIM_ALL_MEMBERS`) support the DAX `ALL()` pattern in scenario 5.

In [ ]:
DATA_SQL = [
 "\nCREATE OR REPLACE TABLE {FQ}.DIM_CALENDAR AS\nWITH d AS (\n  SELECT DATEADD(day, SEQ4(), '2020-01-01'::DATE) AS date_key\n  FROM TABLE(GENERATOR(ROWCOUNT => 1461))\n), f AS (\n  SELECT date_key,\n         2020 + FLOOR(DATEDIFF(day, '2019-12-30'::DATE, date_key) / 364) AS fiscal_year,\n         FLOOR(MOD(DATEDIFF(day, '2019-12-30'::DATE, date_key), 364) / 7) + 1 AS fiscal_week\n  FROM d\n), p AS (\n  SELECT f.*, CEIL(fiscal_week / 13) AS fiscal_quarter,\n         fiscal_week - 13 * (CEIL(fiscal_week / 13) - 1) AS week_in_quarter\n  FROM f\n), q AS (\n  SELECT p.*, CASE WHEN week_in_quarter <= 4 THEN 1 WHEN week_in_quarter <= 8 THEN 2 ELSE 3 END AS period_in_quarter\n  FROM p\n)\nSELECT\n  date_key,\n  YEAR(date_key)                                 AS year,\n  YEAR(date_key) * 100 + MONTH(date_key)         AS year_month_num,\n  TO_CHAR(date_key, 'MON YYYY')                  AS year_month_name,\n  DATE_TRUNC('month', date_key)                  AS month_start,\n  DATE_TRUNC('year', date_key)                   AS year_start,\n  fiscal_year::INT                               AS fiscal_year,\n  ((fiscal_quarter - 1) * 3 + period_in_quarter)::INT AS fiscal_period,\n  'FY' || fiscal_year::INT || ' P' || LPAD(((fiscal_quarter - 1) * 3 + period_in_quarter)::INT, 2, '0') AS fiscal_period_name,\n  DATEADD(day, 364 * (fiscal_year - 2020)\n               + 7 * ((fiscal_quarter - 1) * 13 + CASE period_in_quarter WHEN 1 THEN 0 WHEN 2 THEN 4 ELSE 8 END),\n          '2019-12-30'::DATE)                    AS fiscal_period_start,\n  DATEADD(day, 364 * (fiscal_year - 2020), '2019-12-30'::DATE) AS fiscal_year_start\nFROM q\n",
 "\nCREATE OR REPLACE TABLE {FQ}.DIM_PRODUCT AS\nWITH b AS (\n  SELECT * FROM VALUES\n    ('Laundry',   'Tide',      'Procter & Gamble', 2.40),\n    ('Laundry',   'Persil',    'Henkel',           2.10),\n    ('Hair Care', 'Pantene',   'Procter & Gamble', 1.80),\n    ('Hair Care', 'Dove',      'Unilever',         1.60),\n    ('Oral Care', 'Oral-B',    'Procter & Gamble', 1.20),\n    ('Oral Care', 'Signal',    'Unilever',         0.90),\n    ('Clearance', 'Old Spice', 'Procter & Gamble', -0.70),\n    ('Clearance', 'Somat',     'Henkel',           -0.50)\n    AS t(category, brand, manufacturer, base_margin)\n)\nSELECT\n  ROW_NUMBER() OVER (ORDER BY b.brand, g.n)  AS product_id,\n  b.category, b.brand, b.manufacturer,\n  b.brand || ' SKU ' || g.n                   AS product_name,\n  ROUND(b.base_margin * (0.8 + 0.1 * g.n), 2) AS unit_margin\nFROM b\nCROSS JOIN (SELECT SEQ4() + 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => 5))) g\n",
 "\nCREATE OR REPLACE TABLE {FQ}.DIM_STORE AS\nWITH c AS (\n  SELECT * FROM VALUES\n    ('Poland',  'Biedronka',   1.40), ('Poland',  'Lidl PL',     1.10),\n    ('Poland',  'Rossmann PL', 0.80), ('Poland',  'Zabka',       0.60),\n    ('Germany', 'dm',          1.50), ('Germany', 'Rossmann DE', 1.20),\n    ('Germany', 'Edeka',       1.00), ('Germany', 'Rewe',        0.90),\n    ('UK',      'Tesco',       1.30), ('UK',      'Boots',       0.90),\n    ('UK',      'Superdrug',   0.60)\n    AS t(country, customer, customer_weight)\n)\nSELECT\n  ROW_NUMBER() OVER (ORDER BY c.country, c.customer, g.n) AS store_id,\n  c.customer, c.country,\n  c.customer || ' #' || g.n AS store_name,\n  c.customer_weight\nFROM c\nCROSS JOIN (SELECT SEQ4() + 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => 5))) g\n",
 "\nCREATE OR REPLACE TABLE {FQ}.FACT_SALES AS\nWITH grid AS (\n  SELECT\n    c.date_key, s.store_id, p.product_id, p.unit_margin,\n    s.customer_weight\n      * CASE c.year WHEN 2020 THEN 1.00 WHEN 2021 THEN 1.08 ELSE 1.15 END\n      * (1 + 0.20 * SIN(2 * PI() * DAYOFYEAR(c.date_key) / 365)) AS demand\n  FROM {FQ}.DIM_CALENDAR c\n  CROSS JOIN {FQ}.DIM_STORE s\n  CROSS JOIN {FQ}.DIM_PRODUCT p\n  WHERE c.year <= 2022\n), q AS (\n  SELECT grid.*,\n         ROUND(UNIFORM(40, 160, HASH(date_key, store_id, product_id)) * demand)::INT AS sales_qty\n  FROM grid\n)\nSELECT\n  date_key AS sale_date, store_id, product_id, sales_qty,\n  ROUND(sales_qty * unit_margin + UNIFORM(-5, 5, HASH(date_key, store_id, product_id, 'np')), 2) AS net_profit\nFROM q\n",
 "CREATE OR REPLACE TABLE {FQ}.DIM_COUNTRY AS SELECT DISTINCT country FROM {FQ}.DIM_STORE",
 "CREATE OR REPLACE TABLE {FQ}.DIM_BRAND AS SELECT DISTINCT brand FROM {FQ}.DIM_PRODUCT",
 "CREATE OR REPLACE TABLE {FQ}.DIM_CATEGORY AS SELECT DISTINCT category, 1 AS all_key FROM {FQ}.DIM_PRODUCT",
 "CREATE OR REPLACE TABLE {FQ}.DIM_ALL_MEMBERS AS SELECT 1 AS all_key"
]

for stmt in DATA_SQL:
    session.sql(stmt.replace("{FQ}", FQ)).collect()

session.sql(f"SELECT COUNT(*) AS fact_rows FROM {FQ}.FACT_SALES").to_pandas()

### 3. Create the semantic view

The full definition is below. Each scenario explains the part it uses.

In [ ]:
SV_SQL = "\nCREATE OR REPLACE SEMANTIC VIEW {FQ}.DAX_CONTEXT_SV\n  TABLES (\n    sales AS {FQ}.FACT_SALES PRIMARY KEY (sale_date, store_id, product_id),\n    -- \"Last year\" copies of the fact (calendar year and 4-4-5 fiscal year)\n    sales_ly  AS {FQ}.FACT_SALES PRIMARY KEY (sale_date, store_id, product_id),\n    sales_fly AS {FQ}.FACT_SALES PRIMARY KEY (sale_date, store_id, product_id),\n    -- Denominator copies for DAX ALL(Customer) and ALL(Customer, Category)\n    sales_all_cust     AS {FQ}.FACT_SALES PRIMARY KEY (sale_date, store_id, product_id),\n    sales_all_cust_cat AS {FQ}.FACT_SALES PRIMARY KEY (sale_date, store_id, product_id),\n    calendar      AS {FQ}.DIM_CALENDAR    PRIMARY KEY (date_key),\n    product       AS {FQ}.DIM_PRODUCT     PRIMARY KEY (product_id) COMMENT = 'Category path only; no dimensions',\n    product_attrs AS {FQ}.DIM_PRODUCT     PRIMARY KEY (product_id) COMMENT = 'Product attributes ALL(category) keeps',\n    brand         AS {FQ}.DIM_BRAND       PRIMARY KEY (brand),\n    category      AS {FQ}.DIM_CATEGORY    PRIMARY KEY (category),\n    all_members   AS {FQ}.DIM_ALL_MEMBERS PRIMARY KEY (all_key),\n    store         AS {FQ}.DIM_STORE       PRIMARY KEY (store_id),\n    store_geo     AS {FQ}.DIM_STORE       PRIMARY KEY (store_id) COMMENT = 'Country path only; no dimensions',\n    country       AS {FQ}.DIM_COUNTRY     PRIMARY KEY (country)\n  )\n  RELATIONSHIPS (\n    store_to_country       AS store (country) REFERENCES country,\n    store_geo_to_country   AS store_geo (country) REFERENCES country,\n    product_to_category    AS product (category) REFERENCES category,\n    product_attrs_to_brand AS product_attrs (brand) REFERENCES brand,\n    category_to_all        AS category (all_key) REFERENCES all_members,\n\n    sales_to_calendar      AS sales (sale_date) REFERENCES calendar,\n    sales_to_product       AS sales (product_id) REFERENCES product,\n    sales_to_product_attrs AS sales (product_id) REFERENCES product_attrs,\n    sales_to_store         AS sales (store_id) REFERENCES store,\n\n    sales_ly_to_calendar      AS sales_ly (sale_date_ly) REFERENCES calendar,\n    sales_ly_to_product       AS sales_ly (product_id) REFERENCES product,\n    sales_ly_to_product_attrs AS sales_ly (product_id) REFERENCES product_attrs,\n    sales_ly_to_store         AS sales_ly (store_id) REFERENCES store,\n    sales_fly_to_calendar      AS sales_fly (sale_date_fly) REFERENCES calendar,\n    sales_fly_to_product       AS sales_fly (product_id) REFERENCES product,\n    sales_fly_to_product_attrs AS sales_fly (product_id) REFERENCES product_attrs,\n    sales_fly_to_store         AS sales_fly (store_id) REFERENCES store,\n\n    all_cust_to_calendar      AS sales_all_cust (sale_date) REFERENCES calendar,\n    all_cust_to_product       AS sales_all_cust (product_id) REFERENCES product,\n    all_cust_to_product_attrs AS sales_all_cust (product_id) REFERENCES product_attrs,\n    all_cust_to_store_geo     AS sales_all_cust (store_id) REFERENCES store_geo,\n\n    all_cc_to_calendar      AS sales_all_cust_cat (sale_date) REFERENCES calendar,\n    all_cc_to_all_members   AS sales_all_cust_cat (all_key) REFERENCES all_members,\n    all_cc_to_product_attrs AS sales_all_cust_cat (product_id) REFERENCES product_attrs,\n    all_cc_to_store_geo     AS sales_all_cust_cat (store_id) REFERENCES store_geo\n  )\n  FACTS (\n    sales.sales_qty  AS sales_qty,\n    sales.net_profit AS net_profit,\n    sales_ly.sale_date_ly AS DATEADD('year', 1, sale_date),\n    sales_ly.sales_qty    AS sales_qty,\n    sales_ly.net_profit   AS net_profit,\n    sales_fly.sale_date_fly AS DATEADD('day', 364, sale_date),\n    sales_fly.sales_qty     AS sales_qty,\n    sales_all_cust.sales_qty     AS sales_qty,\n    sales_all_cust_cat.sales_qty AS sales_qty,\n    sales_all_cust_cat.all_key   AS 1,\n    country_qty_fixed AS SUM(sales.sales_qty) FIXED DIMENSIONS (country.country, calendar.year),\n    country_qty_excl_customer AS SUM(sales.sales_qty) EXCLUDE DIMENSIONS (store.customer),\n    country_qty_excl_customer_category AS SUM(sales.sales_qty) EXCLUDE DIMENSIONS (store.customer, category.category)\n  )\n  DIMENSIONS (\n    calendar.date_key            AS date_key,\n    calendar.year                AS year,\n    calendar.year_month_num      AS year_month_num,\n    calendar.year_month_name     AS year_month_name,\n    calendar.month_start         AS month_start,\n    calendar.year_start          AS year_start,\n    calendar.fiscal_year         AS fiscal_year,\n    calendar.fiscal_period_name  AS fiscal_period_name,\n    calendar.fiscal_period_start AS fiscal_period_start,\n    product_attrs.product_id     AS product_id,\n    product_attrs.manufacturer   AS manufacturer,\n    category.category            AS category,\n    brand.brand                  AS brand,\n    store.store_id               AS store_id,\n    store.customer               AS customer,\n    country.country              AS country\n  )\n  METRICS (\n    sales.sales_quantity   AS SUM(sales.sales_qty),\n    sales.net_profit_total AS SUM(sales.net_profit),\n\n    -- time_shift (Private Preview): one metric per anchor grain\n    sales.sales_qty_ya_daily AS sales.sales_quantity WITH time_shift(calendar.date_key, year, -1),\n    sales.sales_qty_ya_fiscal_period AS sales.sales_quantity WITH time_shift(calendar.fiscal_period_start, week, -52),\n\n    -- \"Last year\" copy of the fact: one metric, any grain\n    sales_ly.sales_qty_ly   AS SUM(sales_ly.sales_qty),\n    sales_ly.net_profit_ly  AS SUM(sales_ly.net_profit),\n    sales_fly.sales_qty_fly AS SUM(sales_fly.sales_qty),\n    sales_qty_yoy_pct AS DIV0(sales.sales_quantity - sales_ly.sales_qty_ly, sales_ly.sales_qty_ly),\n\n    -- DAX ALL(...): coarse-key denominators\n    sales_all_cust.country_qty_all_customers    AS SUM(sales_all_cust.sales_qty),\n    sales_all_cust_cat.country_qty_all_cust_cat AS SUM(sales_all_cust_cat.sales_qty),\n    customer_share_all AS DIV0(sales.sales_quantity, sales_all_cust.country_qty_all_customers),\n    cust_cat_share_all AS DIV0(sales.sales_quantity, sales_all_cust_cat.country_qty_all_cust_cat),\n\n    -- DAX ALLEXCEPT (FIXED) and ALLSELECTED (EXCLUDE): LOD metrics (Public Preview)\n    country_total_allexcept   AS AVG(country_qty_fixed),\n    country_total_allselected AS AVG(country_qty_excl_customer),\n    customer_share_allexcept   AS SUM(sales.sales_qty) / AVG(country_qty_fixed),\n    customer_share_allselected AS SUM(sales.sales_qty) / AVG(country_qty_excl_customer),\n    cust_cat_share_allselected AS SUM(sales.sales_qty) / AVG(country_qty_excl_customer_category)\n  )\n  COMMENT = 'DAX context manipulation patterns (Year Ago, ALL, ALLEXCEPT, ALLSELECTED) on a Snowflake semantic view'\n"

session.sql(SV_SQL.replace("{FQ}", FQ)).collect()
print(f"Created {SV}")

## Warm-up: query the semantic view like a table

Semantic views support Standard SQL: select dimensions, wrap metrics in `AGG()`, and group by the dimensions.

In [ ]:
SELECT country, year, AGG(sales_quantity) AS sales_qty
FROM {{SV}}
WHERE year <= 2022
GROUP BY country, year
ORDER BY country, year

## Scenario 1: Year Ago at the date grain

The article's first measure. `time_shift` shifts a metric along a time dimension:

```sql
sales.sales_qty_ya_daily AS sales.sales_quantity
  WITH time_shift(calendar.date_key, year, -1)
```

In [ ]:
SELECT date_key, AGG(sales_quantity) AS sales_qty, AGG(sales_qty_ya_daily) AS sales_qty_ya
FROM {{SV}}
WHERE date_key BETWEEN '2020-02-25' AND '2021-03-03'
GROUP BY date_key
HAVING date_key BETWEEN '2021-02-25' AND '2021-03-03'
ORDER BY date_key

## Scenario 2: Year Ago at any grain

The article hits two breaks. Adding a second calendar column breaks the daily measure, and querying it by month
returns only the last day of the month. `time_shift` hits the same first break: it partitions by every other
dimension in the query, so "Feb 2021" can't see "Feb 2020".

In [ ]:
SELECT date_key, year_month_name, AGG(sales_quantity) AS sales_qty, AGG(sales_qty_ya_daily) AS sales_qty_ya
FROM {{SV}}
WHERE date_key BETWEEN '2020-02-01' AND '2021-03-03'
GROUP BY date_key, year_month_name
HAVING date_key BETWEEN '2021-02-26' AND '2021-03-02'
ORDER BY date_key

The second break fails safely. Asking for the daily measure by month is a compile error, not a misleading number:

In [ ]:
try:
    session.sql(f"""
        SELECT month_start, AGG(sales_qty_ya_daily)
        FROM {SV}
        GROUP BY month_start
    """).collect()
except Exception as e:
    print(str(e).splitlines()[-1])

### The fix: shift the rows, not the measure

This is how DAX `SAMEPERIODLASTYEAR` works through filter context. The semantic view gets a second logical copy of
the fact table (`sales_ly`) whose date key is moved forward one year. Last year's rows then sit on this year's
calendar rows, and `sales_qty_ly` is an ordinary metric that any calendar grouping or filter applies to.

```sql
TABLES        ( sales_ly AS FACT_SALES PRIMARY KEY (sale_date, store_id, product_id) )
FACTS         ( sales_ly.sale_date_ly AS DATEADD('year', 1, sale_date) )
RELATIONSHIPS ( sales_ly_to_calendar AS sales_ly (sale_date_ly) REFERENCES calendar,
                sales_ly_to_store    AS sales_ly (store_id)     REFERENCES store, ... )
METRICS       ( sales_ly.sales_qty_ly AS SUM(sales_ly.sales_qty) )
```

One metric at three grains. Note Feb 28, 2021: it picks up Feb 28 **and** Feb 29, 2020, as DAX `DATEADD` does at
month end.

In [ ]:
SELECT date_key, year_month_name, AGG(sales_quantity) AS sales_qty, AGG(sales_qty_ly) AS sales_qty_ly
FROM {{SV}}
GROUP BY date_key, year_month_name
HAVING date_key BETWEEN '2021-02-26' AND '2021-03-02'
ORDER BY date_key

In [ ]:
SELECT month_start, AGG(sales_quantity) AS sales_qty, AGG(sales_qty_ly) AS sales_qty_ly,
       ROUND(AGG(sales_qty_yoy_pct) * 100, 1) AS yoy_pct
FROM {{SV}}
GROUP BY month_start
HAVING month_start BETWEEN '2021-01-01' AND '2021-04-01'
ORDER BY month_start

In [ ]:
SELECT year, AGG(sales_quantity) AS sales_qty, AGG(sales_qty_ly) AS sales_qty_ly,
       ROUND(AGG(sales_qty_yoy_pct) * 100, 1) AS yoy_pct
FROM {{SV}}
GROUP BY year
HAVING year <= 2022
ORDER BY year

## Scenario 3: 4-4-5 fiscal calendar

The article notes that its `GREATEST()` workaround fails with 4-4-5 months. A 4-4-5 fiscal year is 52 weeks
(364 days), so shift by 364 days: either a second "last fiscal year" copy of the fact (`sales_qty_fly`), or
`time_shift(fiscal_period_start, week, -52)`.

In [ ]:
WITH ts AS (
  SELECT fiscal_period_start, AGG(sales_quantity) AS sales_qty, AGG(sales_qty_ya_fiscal_period) AS ya_time_shift
  FROM {{SV}} GROUP BY fiscal_period_start
), ly AS (
  SELECT fiscal_period_start, fiscal_period_name, AGG(sales_qty_fly) AS ya_fiscal_copy
  FROM {{SV}} GROUP BY fiscal_period_start, fiscal_period_name
)
SELECT ly.fiscal_period_name, ts.fiscal_period_start, ts.sales_qty, ts.ya_time_shift, ly.ya_fiscal_copy
FROM ts JOIN ly USING (fiscal_period_start)
WHERE ts.fiscal_period_start BETWEEN '2021-01-25' AND '2021-05-24'
ORDER BY ts.fiscal_period_start

## Scenario 4: metrics that go negative

The article's other `GREATEST()` caveat: it assumes the whole-period value is the largest. For the Clearance category,
net profit is negative, so one day is "greater" than the whole month. This cell rebuilds the article's Databricks
flavors from the base tables (a daily window queried by month returns the last day) and compares them with the
"last year" metric.

In [ ]:
WITH sv AS (
  SELECT month_start, AGG(net_profit_total) AS net_profit, AGG(net_profit_ly) AS snowflake_ly
  FROM {{SV}}
  WHERE category = 'Clearance'
  GROUP BY month_start
), base AS (
  SELECT f.sale_date, SUM(f.net_profit) AS np
  FROM {{FQ}}.FACT_SALES f JOIN {{FQ}}.DIM_PRODUCT p USING (product_id)
  WHERE p.category = 'Clearance'
  GROUP BY 1
), sim AS (
  SELECT DATEADD('year', 1, DATE_TRUNC('month', sale_date)) AS month_start,
         SUM(np) AS monthly_ya, MAX_BY(np, sale_date) AS daily_ya_last_day
  FROM base GROUP BY 1
)
SELECT sv.month_start,
       ROUND(sim.daily_ya_last_day)                        AS databricks_daily_ya,
       ROUND(sim.monthly_ya)                               AS databricks_monthly_ya,
       ROUND(GREATEST(sim.daily_ya_last_day, sim.monthly_ya)) AS databricks_greatest,
       ROUND(sv.snowflake_ly)                              AS snowflake_ly
FROM sv JOIN sim USING (month_start)
WHERE sv.month_start BETWEEN '2021-01-01' AND '2021-04-01'
ORDER BY sv.month_start

## Scenarios 5–7: ALL, ALLSELECTED, ALLEXCEPT (customer share of country)

Same test as the article: remove one customer per country. Add a month and a brand filter too, because
`ALL(Store[Customer])` must ignore the customer filter but **keep** every other filter.

| Metric | Construct | DAX |
|---|---|---|
| `customer_share_all` | Coarse-key denominator (below) | `ALL(Store[Customer])` |
| `customer_share_allselected` | `SUM(qty) EXCLUDE DIMENSIONS (store.customer)` | `ALLSELECTED(Store[Customer])` |
| `customer_share_allexcept` | `SUM(qty) FIXED DIMENSIONS (country.country, calendar.year)` | `ALLEXCEPT(Store, Store[Country], Calendar[Year])` |

**How `ALL` works.** The denominator `sales_all_cust` is another copy of the fact. It never joins the `store` table
that carries `customer`. It joins a second copy, `store_geo`, and through it the one-row-per-country table. So a
customer filter can reach the denominator only through Country, a one-to-many hop. A filter across that hop keeps a
country's rows when *any* of its customers pass, so filtered-out customers stay in the denominator. Date and brand
still join directly and filter as usual.

```sql
store_to_country      AS store (country) REFERENCES country,
store_geo_to_country  AS store_geo (country) REFERENCES country,
all_cust_to_store_geo AS sales_all_cust (store_id) REFERENCES store_geo,
...
customer_share_all AS DIV0(sales.sales_quantity, sales_all_cust.country_qty_all_customers)
```

In [ ]:
SELECT country, customer,
       AGG(sales_quantity)                         AS sales_qty,
       AGG(country_qty_all_customers)              AS denom_all,
       ROUND(AGG(customer_share_all), 4)           AS share_all,
       AGG(country_total_allselected)              AS denom_allselected,
       ROUND(AGG(customer_share_allselected), 4)   AS share_allselected
FROM {{SV}}
WHERE customer NOT IN ('Zabka', 'Rewe', 'Superdrug')
  AND month_start = '2021-02-01'
  AND brand <> 'Somat'
GROUP BY country, customer
ORDER BY country, customer

In [ ]:
# ALL shares are of the whole country; ALLSELECTED shares are of the visible customers (100% per country)
s5_s6_shares.groupby("COUNTRY")[["SHARE_ALL", "SHARE_ALLSELECTED"]].sum().round(4)

### Scenario 7 (bonus): why FIXED isn't ALL

`FIXED` calculates before filters apply, so it ignores **every** filter on a column outside its list, not just the
customer filter. That's `ALLEXCEPT`. Filter to one month and one customer out: `FIXED(country, year)` still returns
the full-year total, while the `ALL` denominator returns the February total for all customers.

In [ ]:
SELECT country,
       AGG(sales_quantity)            AS sales_qty_feb_excl_tesco,
       AGG(country_qty_all_customers) AS dax_all,
       AGG(country_total_allexcept)   AS dax_allexcept_fixed
FROM {{SV}}
WHERE month_start = '2021-02-01' AND customer <> 'Tesco'
GROUP BY country
ORDER BY country

## Scenario 8: customer × category share of country

The article's two-column analogy. For `ALLSELECTED`, list both columns in `EXCLUDE`. For `ALL`, there's no natural
"coarser than category" column, so the denominator reaches Category through a constant key: a computed fact
`all_key AS 1` referencing the one-row `DIM_ALL_MEMBERS`, which every category also references. Product attributes
that must stay filterable (product, manufacturer, brand) sit on a separate `product_attrs` copy that joins directly.

Filter to Laundry and to two brands, one of which (Dove) is Hair Care. `ALL(Customer, Category)` drops the category
filter but keeps the brand filter, so Dove counts in the denominator, just as in DAX.

In [ ]:
SELECT country,
       AGG(sales_quantity)            AS laundry_qty_tide_dove,
       AGG(country_qty_all_cust_cat)  AS denom_all_cust_cat,
       ROUND(AGG(cust_cat_share_all), 4) AS share_all
FROM {{SV}}
WHERE year = 2021 AND brand IN ('Tide', 'Dove') AND category = 'Laundry' AND customer <> 'Tesco'
GROUP BY country
ORDER BY country

## Bonus: ask it in plain English

The "last year" and ALL metrics are ordinary metrics, so Cortex Analyst can use them. Point Cortex Analyst (or a
Cortex Agent) at the semantic view and ask: *"For February 2021, show each customer's sales quantity, sales quantity
last year, and share of its country's total across all customers."* (`time_shift` and LOD metrics aren't used by
Cortex Analyst yet.)

## Clean up

When you're done, drop the demo schema:

In [ ]:
# session.sql(f"DROP SCHEMA IF EXISTS {FQ}").collect()